<a href="https://colab.research.google.com/github/Gus19-8/arq.-calculadora/blob/main/proyectoU3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from IPython.display import display

# ==========================================
# 1. FUNCIONES DE CÁLCULO (Lógica pura)
# ==========================================

def calcular_subtotal(cantidad: float, precio_unitario: float) -> float:
    """
    Calcula el costo subtotal de un material.
    Lanza un ValueError si alguno de los valores es negativo.
    """
    if cantidad < 0 or precio_unitario < 0:
        raise ValueError("La cantidad y el precio unitario no pueden ser negativos.")
    return round(cantidad * precio_unitario, 2)


def generar_presupuesto(lista_materiales: list) -> tuple:
    """
    Recibe una lista de diccionarios con los datos de cada material,
    genera un DataFrame de pandas con los subtotales y calcula el costo total.
    """
    if not lista_materiales:
        df_vacio = pd.DataFrame(
            columns=["Material", "Unidad", "Cantidad", "Precio Unitario ($)", "Subtotal ($)"]
        )
        return df_vacio, 0.0

    filas = []
    costo_total = 0.0

    for item in lista_materiales:
        subtotal = calcular_subtotal(item["cantidad"], item["precio_unitario"])
        costo_total += subtotal

        filas.append({
            "Material": item["material"],
            "Unidad": item["unidad"],
            "Cantidad": item["cantidad"],
            "Precio Unitario ($)": round(item["precio_unitario"], 2),
            "Subtotal ($)": subtotal
        })

    df_presupuesto = pd.DataFrame(filas)
    return df_presupuesto, round(costo_total, 2)


# ==========================================
# 2. INTERFAZ INTERACTIVA PARA EL USUARIO
# ==========================================

def ejecutar_calculadora():
    """
    Solicita los materiales al usuario de forma dinámica mediante consola
    y muestra la tabla final de presupuesto en Google Colab.
    """
    print("=== CALCULADORA DE PRESUPUESTO DE MATERIALES ===")
    print("Ingresa los datos de cada material. Escribe 'fin' en el nombre para terminar.\n")

    materiales_ingresados = []

    while True:
        nombre = input("Nombre del material (o 'fin' para calcular): ").strip()

        if nombre.lower() == "fin":
            break

        if not nombre:
            print("⚠️ El nombre del material no puede estar vacío. Intenta de nuevo.\n")
            continue

        unidad = input(f"Unidad de medida para '{nombre}' (ej. piezas, bultos, litros, m³): ").strip()
        if not unidad:
            unidad = "unidades"

        try:
            cantidad = float(input(f"Cantidad de '{nombre}' ({unidad}): "))
            precio = float(input(f"Precio unitario por {unidad} ($): "))

            if cantidad <= 0 or precio < 0:
                print("⚠️ Error: La cantidad debe ser mayor a 0 y el precio no puede ser negativo.\n")
                continue

            materiales_ingresados.append({
                "material": nombre,
                "unidad": unidad,
                "cantidad": cantidad,
                "precio_unitario": precio
            })
            print(f"✅ '{nombre}' agregado correctamente.\n")

        except ValueError:
            print("⚠️ Error: Por favor ingresa únicamente valores numéricos válidos para cantidad y precio.\n")

    # Generar y mostrar resultados
    print("\n" + "=" * 50)
    if not materiales_ingresados:
        print("No se ingresaron materiales para calcular.")
    else:
        tabla_final, total_obra = generar_presupuesto(materiales_ingresados)
        print("📋 RESUMEN DEL PRESUPUESTO DE OBRA:\n")
        display(tabla_final)
        print(f"\n💰 COSTO TOTAL DE LOS MATERIALES: ${total_obra:,.2f}")
    print("=" * 50)


# Ejecutar el programa principal
if __name__ == "__main__":
    ejecutar_calculadora()

=== CALCULADORA DE PRESUPUESTO DE MATERIALES ===
Ingresa los datos de cada material. Escribe 'fin' en el nombre para terminar.

Nombre del material (o 'fin' para calcular): Ladrillo
Unidad de medida para 'Ladrillo' (ej. piezas, bultos, litros, m³): 500 piezas
Cantidad de 'Ladrillo' (500 piezas): 500
Precio unitario por 500 piezas ($): 6
✅ 'Ladrillo' agregado correctamente.

Nombre del material (o 'fin' para calcular): Agua
Unidad de medida para 'Agua' (ej. piezas, bultos, litros, m³): Litros
Cantidad de 'Agua' (Litros): 200
Precio unitario por Litros ($): 5
✅ 'Agua' agregado correctamente.

Nombre del material (o 'fin' para calcular): Fin

📋 RESUMEN DEL PRESUPUESTO DE OBRA:



,Material,Unidad,Cantidad,Precio Unitario ($),Subtotal ($)
0,Ladrillo,500 piezas,500.0,6.0,3000.0
1,Agua,Litros,200.0,5.0,1000.0



💰 COSTO TOTAL DE LOS MATERIALES: $4,000.00


In [ ]:
import unittest

class TestCalculadoraMateriales(unittest.TestCase):

    def test_calcular_subtotal_normal(self):
        """Verifica el cálculo estándar de cantidad por precio."""
        resultado = calcular_subtotal(500, 12.50)
        self.assertEqual(resultado, 6250.00)

    def test_calcular_subtotal_cero(self):
        """Verifica que valores en cero devuelvan 0.0 sin errores."""
        self.assertEqual(calcular_subtotal(0, 230.0), 0.0)
        self.assertEqual(calcular_subtotal(50, 0), 0.0)

    def test_calcular_subtotal_negativo(self):
        """Verifica que cantidades o precios negativos lancen ValueError."""
        with self.assertRaises(ValueError):
            calcular_subtotal(-10, 50.0)
        with self.assertRaises(ValueError):
            calcular_subtotal(10, -50.0)

    def test_generar_presupuesto_varios_materiales(self):
        """Verifica la creación de la tabla y la suma total de múltiples materiales."""
        datos_prueba = [
            {"material": "Ladrillos", "unidad": "piezas", "cantidad": 500, "precio_unitario": 10.0},
            {"material": "Cemento gris", "unidad": "bultos", "cantidad": 50, "precio_unitario": 220.0},
            {"material": "Agua", "unidad": "litros", "cantidad": 200, "precio_unitario": 0.50}
        ]
        # Subtotales esperados: 5000 + 11000 + 100 = 16100.0
        df, total = generar_presupuesto(datos_prueba)

        self.assertEqual(len(df), 3)
        self.assertEqual(total, 16100.00)
        self.assertEqual(df.loc[1, "Subtotal ($)"], 11000.00)

    def test_generar_presupuesto_vacio(self):
        """Verifica el comportamiento cuando no se envían materiales."""
        df, total = generar_presupuesto([])
        self.assertTrue(df.empty)
        self.assertEqual(total, 0.0)

# Configuración especial para ejecutar unittest dentro de Google Colab sin errores de sistema
unittest.main(argv=['first-arg-is-ignored'], exit=False)

.....
----------------------------------------------------------------------
Ran 5 tests in 0.016s

OK
